# 27a · A2A: the task lifecycle, streaming and delegating to another agent
Level L5 · Part 5 · professional · ~25 min · builds on 27

## Goal
Send a task to the refunds agent from lesson 27, follow its states (submitted, working, input-required, completed,
failed, rejected, canceled), read its streamed updates and artifacts, and handle a failure.

## Why it matters
Lesson 27 let the support agent find the refunds agent and read its card. It did not send any work. A refund is not one
function call: the refunds agent may need the order number, may refuse under its policy, may fail in payments, and the
support agent may need to cancel it. Without a shared lifecycle, each team invents its own status words and its own
polling loop. Here is what the support agent writes for two teams:

In [1]:
# Without a shared lifecycle: one caller loop per team, each with its own status words.
TEAM_A_DONE = {"done", "error"}  # team A: finished means done or error
TEAM_B_DONE = {"complete"}  # team B: finished means complete; its failures say "failed"
for team, done, status in [("A", TEAM_A_DONE, "error"), ("B", TEAM_B_DONE, "failed")]:
    print(f"team {team}: is '{status}' finished? {status in done}")
print("team B's loop keeps polling after a failure. Nothing in either API told the caller what 'failed' means.")

team A: is 'error' finished? True
team B: is 'failed' finished? False
team B's loop keeps polling after a failure. Nothing in either API told the caller what 'failed' means.


| Without a shared lifecycle | With A2A tasks |
|---|---|
| Each team picks its own status words and loop | Eight states, one meaning each, from the protocol |
| A failure can look like "still working" forever | `failed`, `rejected` and `canceled` end the task |
| Cancel means killing a process and hoping | A cancel request moves the task to `canceled` |

**What it makes easy:** one caller loop works for any agent that speaks A2A.
**When you don't need it:** a call that returns in a second inside your own process. A plain function is enough.

## The idea
An **A2A task** is the unit of work the remote agent owns. The caller sends a user **message**. The agent creates a
**task** with an id and a **context id** (the conversation it belongs to), then moves it through **states**. Each state
change is an event. The agent's words go in messages; its results go in **artifacts** (the refund receipt).
**Streaming** sends events as they happen, so the caller sees progress without polling.

The states: **submitted** (accepted), **working** (running), **input-required** (paused for the caller's answer on the
same task), **auth-required** (paused for sign-in, lesson 26), **completed**, **failed** (the work broke, with a reason),
**rejected** (the agent declined under its policy) and **canceled**. The last four are **terminal**: a retry is a new task.

Analogy: a repair ticket at a phone shop. "Received", "in progress", "we need your PIN", "fixed", "cannot be repaired",
"cancelled by you". Everyone reads the same ticket.

## Picture
```mermaid
stateDiagram-v2
  [*] --> submitted
  submitted --> working
  working --> input_required: needs the order number
  input_required --> working: caller answers on the same task
  working --> completed: receipt artifact
  working --> failed: payments error
  working --> rejected: policy says no
  working --> canceled: caller cancels
  completed --> [*]
  failed --> [*]
  rejected --> [*]
  canceled --> [*]
```

## Step by step
**Step 1 · the refunds agent.** The server is `apps/27a/refunds_agent.py`. It serves lesson 27's card and a task endpoint.
The executor is the agent's code: it reads the caller's message and publishes each state change. The first event for
a new task must be the Task itself. Read `execute` once: it is the whole lifecycle. The sign-in check is left out
here (lesson 26 covers it), so this is a lab server on 127.0.0.1 only.

In [2]:
from pathlib import Path

Path("../apps/27a").mkdir(parents=True, exist_ok=True)  # the server file goes here
Path("../data/27a").mkdir(parents=True, exist_ok=True)  # run-time logs go here

**The server file.** The cell below writes it into `apps/27a/`. Read it once; the next steps run it.

In [3]:
%%writefile ../apps/27a/refunds_agent.py
"""The Brightside refunds agent as an A2A server (lesson 27a): the lesson 27 card plus a task endpoint.
Run: python refunds_agent.py <port>. A lab server on 127.0.0.1 with no sign-in check (lesson 26 covers sign-in)."""
import asyncio
import json
import re
import sys
from pathlib import Path

import uvicorn
from a2a.helpers.proto_helpers import new_task_from_user_message, new_text_part
from a2a.server.agent_execution import AgentExecutor
from a2a.server.request_handlers import DefaultRequestHandlerV2
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.types import AgentCapabilities, AgentCard, AgentInterface, AgentSkill, TaskState
from a2a.utils.constants import PROTOCOL_VERSION_CURRENT, TransportProtocol
from starlette.applications import Starlette

ORDERS = json.loads((Path(__file__).resolve().parents[2] / "data" / "support_eval.json").read_text())["world"]["orders"]
STEPS = ["Looking up the order", "Checking the refund policy", "Sending the refund to payments", "Writing the receipt"]


def build_card(port: int) -> AgentCard:
    skill = AgentSkill(id="issue_refund", name="Issue refund", tags=["refund", "write"],
                       description="Refund a lost or delivered order, once. Send the order number.",
                       examples=["Refund order 1003, it was lost"])
    return AgentCard(
        name="Brightside refunds agent", description="Brightside Outfitters: refund requests.", version="1.0.0",
        supported_interfaces=[AgentInterface(url=f"http://127.0.0.1:{port}/", protocol_binding=TransportProtocol.JSONRPC,
                                             protocol_version=PROTOCOL_VERSION_CURRENT)],
        capabilities=AgentCapabilities(streaming=True, push_notifications=False),
        default_input_modes=["text/plain"], default_output_modes=["text/plain"], skills=[skill])


class RefundExecutor(AgentExecutor):
    """One refund per task. Each state change is one event on the queue; the SDK streams the events to the caller."""

    async def execute(self, context, event_queue):
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        if context.current_task is None:  # a new task: the Task object must be the first event
            await event_queue.enqueue_event(new_task_from_user_message(context.message))  # SUBMITTED
        await updater.start_work()  # WORKING
        match = re.search(r"\b(\d{4})\b", context.get_user_input())
        if match is None:  # pause: the caller answers on the same task
            await updater.requires_input(updater.new_agent_message([new_text_part("Which order number is it? Reply with four digits, such as 1003.")]))
            return
        order_id = match.group(1)
        if order_id not in ORDERS:  # the work failed: a terminal state with the reason
            await updater.failed(updater.new_agent_message([new_text_part(f"Order {order_id} is not in the orders system.")]))
            return
        order = ORDERS[order_id]
        if order["status"] not in ("lost", "delivered"):  # a policy answer, not an error
            await updater.reject(updater.new_agent_message([new_text_part(f"Order {order_id} is {order['status']}; a person must review it.")]))
            return
        for step in STEPS:  # progress updates the caller can stream
            await updater.update_status(TaskState.TASK_STATE_WORKING, updater.new_agent_message([new_text_part(step)]))
            await asyncio.sleep(0.4)
        await updater.add_artifact([new_text_part(f"Refund receipt: order {order_id}, {order['total']:.2f} USD, refunded once.")],
                                   name="refund-receipt")
        await updater.complete()

    async def cancel(self, context, event_queue):
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        await updater.cancel(updater.new_agent_message([new_text_part("Refund cancelled before payment.")]))


if __name__ == "__main__":
    port = int(sys.argv[1])
    handler = DefaultRequestHandlerV2(agent_executor=RefundExecutor(), task_store=InMemoryTaskStore(), agent_card=build_card(port))
    routes = create_agent_card_routes(build_card(port)) + create_jsonrpc_routes(handler, rpc_url="/")
    uvicorn.run(Starlette(routes=routes), host="127.0.0.1", port=port, log_level="warning")

Overwriting ../apps/27a/refunds_agent.py


**Step 2 · start the agent.** It runs in its own process on port 9270 (this lesson's port, `progress.py --ports 27a`).
The notebook checks the port is free, starts the agent and polls its card until it answers HTTP 200. Logs go to
`../data/27a/`. The helper functions come first.

In [4]:
import socket
import subprocess
import sys
import time

import httpx

SERVER = Path("../apps/27a/refunds_agent.py").resolve()
REFUNDS_URL = "http://127.0.0.1:9270"


def port_open(port: int) -> bool:
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

The start cell below stops early if the port is busy, starts the agent and waits for its card.

In [5]:
if port_open(9270):
    raise RuntimeError("port 9270 is busy: stop the old process first")
log = open("../data/27a/refunds.log", "w")  # server errors go here, not into the notebook
server = subprocess.Popen([sys.executable, str(SERVER), "9270"], stdout=log, stderr=subprocess.STDOUT)
for _ in range(100):  # poll for up to 10 seconds; stop early if the process died
    try:
        if httpx.get(f"{REFUNDS_URL}/.well-known/agent-card.json", timeout=0.5).status_code == 200:
            break
    except httpx.HTTPError:
        pass
    if server.poll() is not None:
        break
    time.sleep(0.1)
print("card answers:", httpx.get(f"{REFUNDS_URL}/.well-known/agent-card.json").status_code)

card answers: 200


**Step 3 · send a task and watch its states.** The client is `create_client` from the SDK, with streaming on. Each
streamed event is one printed line: a state, an artifact or the task itself. Note `role=Role.ROLE_USER`: a caller's
message comes from the user.

In [6]:
from a2a.client import ClientConfig, create_client
from a2a.helpers.proto_helpers import new_text_message
from a2a.types import Role, SendMessageRequest, TaskState


def state_name(state: int) -> str:
    return TaskState.Name(state).removeprefix("TASK_STATE_").lower().replace("_", "-")


def describe(event) -> str:
    """One line per streamed event: the state with the agent's words, or the artifact text."""
    if event.HasField("task"):
        return f"task {event.task.id[:8]} · {state_name(event.task.status.state)}"
    if event.HasField("status_update"):
        status = event.status_update.status
        words = status.message.parts[0].text if status.HasField("message") and status.message.parts else ""
        return f"  {state_name(status.state):<15}{words}"
    if event.HasField("artifact_update"):
        return f"  artifact: {event.artifact_update.artifact.parts[0].text}"
    return "  (other event)"

`send` does the rest: it sends the message, prints each event and keeps the last state it saw.

In [7]:
async def send(client, text: str, task_id: str = "", context_id: str = "") -> tuple[str, str, str]:
    """Send one user message, print each streamed event, return (task id, context id, last state)."""
    request = SendMessageRequest(message=new_text_message(text, role=Role.ROLE_USER,
                                                          task_id=task_id or None, context_id=context_id or None))
    last = ("", "", "")
    async for event in client.send_message(request):
        print(describe(event))
        if event.HasField("task"):
            last = (event.task.id, event.task.context_id, state_name(event.task.status.state))
        elif event.HasField("status_update"):
            s = event.status_update
            last = (s.task_id, s.context_id, state_name(s.status.state))
    return last


client = await create_client(REFUNDS_URL, ClientConfig(streaming=True))  # the card tells the client how to call
t1, c1, state = await send(client, "Refund order 1003, it was lost")
print("final state:", state)

task 2473894d · submitted
  working        
  working        Looking up the order


  working        Checking the refund policy


  working        Sending the refund to payments


  working        Writing the receipt


  artifact: Refund receipt: order 1003, 89.50 USD, refunded once.
  completed      
final state: completed


Read the output. The task starts as `submitted`, moves to `working` and streams each step, then `completed` arrives
with the receipt as an artifact. The stream ends after the terminal state.

**Step 4 · pause for input, then resume.** "Refund it please" has no order number, so the agent asks and the task
waits in `input-required`. The caller answers on the **same task id and context id**, and the agent finishes that task.

In [8]:
t2, c2, state = await send(client, "Refund it please")
print("paused in:", state, "| task", t2[:8])
t2, c2, state = await send(client, "Order 1003", t2, c2)
print("resumed, final state:", state, "| task", t2[:8])

task 83c884f1 · submitted
  working        
  input-required Which order number is it? Reply with four digits, such as 1003.
paused in: input-required | task 83c884f1
  working        
  working        Looking up the order


  working        Checking the refund policy


  working        Sending the refund to payments


  working        Writing the receipt


  artifact: Refund receipt: order 1003, 89.50 USD, refunded once.
  completed      
resumed, final state: completed | task 83c884f1


**Step 5 · failure and rejection.** Order 9999 is not in the orders system: the work fails, with a reason. Order 1002
is in transit, so the policy says no: the task is `rejected`. Both are terminal, and both carry a reason the caller can
show or log.

In [9]:
for text in ["Refund order 9999", "Refund order 1002, it arrived broken"]:
    _, _, state = await send(client, text)
    print("=>", state, "\n")

task 76a8cd83 · submitted
  working        
  failed         Order 9999 is not in the orders system.
=> failed 

task bafb999f · submitted
  working        
  rejected       Order 1002 is in transit; a person must review it.
=> rejected 



**Step 6 · cancel a task in progress.** The caller starts a refund, waits for the first step with words, and sends a
cancel request for that task. The agent's `cancel` method publishes `canceled`, and the stored task keeps that state.

In [10]:
from a2a.types import CancelTaskRequest, GetTaskRequest

cancel_id = ""
async for event in client.send_message(SendMessageRequest(message=new_text_message("Refund order 1003, it was lost", role=Role.ROLE_USER))):
    print(describe(event))
    if event.HasField("status_update") and event.status_update.status.HasField("message") and not cancel_id:
        cancel_id = event.status_update.task_id
        await client.cancel_task(CancelTaskRequest(id=cancel_id))
stored = await client.get_task(GetTaskRequest(id=cancel_id))
print("stored state after cancel:", state_name(stored.status.state), "| receipts:", len(stored.artifacts))

task 95934d4c · submitted
  working        
  working        Looking up the order
  canceled       Refund cancelled before payment.
stored state after cancel: canceled | receipts: 0


**Step 7 · what the caller does with each state.** The support agent needs one action per state. A state with no action
is a loop that never ends, so the table has all of them.

In [11]:
NEXT_STEP = {
    "submitted": "wait: the agent has the task",
    "working": "wait, and show the progress line to the customer",
    "input-required": "ask the customer the question in the message; answer on the same task",
    "auth-required": "send the customer to sign in (lesson 26); then continue on the same task",
    "completed": "read the receipt artifact and tell the customer the refund is done",
    "failed": "tell the customer, log the reason; a retry is a NEW task",
    "rejected": "open a ticket for a person; do not retry",
    "canceled": "tell the customer the refund was stopped",
}
for state in NEXT_STEP:
    print(f"{state:<15}-> {NEXT_STEP[state]}")

submitted      -> wait: the agent has the task
working        -> wait, and show the progress line to the customer
input-required -> ask the customer the question in the message; answer on the same task
auth-required  -> send the customer to sign in (lesson 26); then continue on the same task
completed      -> read the receipt artifact and tell the customer the refund is done
failed         -> tell the customer, log the reason; a retry is a NEW task
rejected       -> open a ticket for a person; do not retry
canceled       -> tell the customer the refund was stopped


A caller can also fail before any task exists: the agent is down or the URL is stale. The card fetch raises an error
that the caller catches, the same way lesson 27 skipped a stale registry entry.

In [12]:
from a2a.client.errors import AgentCardResolutionError

try:
    await create_client("http://127.0.0.1:9279", ClientConfig(streaming=True))  # 9279: nothing listens here
except AgentCardResolutionError as err:
    print("no task was sent; the caller reports this and tries another agent:", str(err)[:60], "…")

no task was sent; the caller reports this and tries another agent: Network communication error fetching agent card from http:// …


## At work
- Long agent work runs as a task with a state machine, not one HTTP call. Dashboards and retries are built on these
  states. Retries and idempotency: lesson 32.
- Teams trace each state change and each tool step, so an operator sees where a refund waited. → lesson 29.
- A refund is a write. Policy checks on the arguments and a human approval come before payment. → lessons 14 and 31.
- Go deeper: push notifications for tasks that run for hours, signed cards, and durable runtimes such as Temporal that
  keep a task alive across restarts.

## Common mistakes
**Mistake 1 · the message from the wrong side.** `new_text_message` makes an agent message unless you pass
`role=Role.ROLE_USER`. The agent refuses it. The cell below fails on purpose.

In [13]:
async def wrong_role():
    async for _ in client.send_message(SendMessageRequest(message=new_text_message("Refund order 1003"))):
        pass


await wrong_role()

InternalError: Message must be from a user

The fix: pass `role=Role.ROLE_USER` on every message the caller sends.

**Mistake 2 · a follow-up to a finished task.** A terminal task never reopens. The cell below sends a message to the
completed task from step 3 and fails on purpose.

In [14]:
await send(client, "Order 1003 again", t1, c1)

UnsupportedOperationError: Task 2473894d-17c5-4bd6-956b-0b59e4b1bdc4 is in terminal state: TASK_STATE_COMPLETED

The fix: send a new message without the old task id. The agent starts a new task, and the caller keeps both ids.

**Mistake 3 · waiting only for `completed`.** A task that needs input never reaches `completed`. A loop that waits for
it hangs. Handle every state in the table from step 7.

## Production checklist
- Store tasks and their events in a database (the SDK ships `DatabaseTaskStore`). The in-memory store used here loses
  every task when the process restarts.
- Make each write safe to repeat (lesson 32): a retry of "refund order 1003" must not refund twice.
- Set a deadline on the caller. On timeout, send a cancel, read the stored state, and only then tell the customer.
- Keep the task id and context id in your own system, next to the case, so you can find the task after a restart.
- Require sign-in on every request (lesson 26). Never put customer data in a status message another team can read.
- Log each state change with the task id and context id (lesson 29).

## Try it
Write `final_state(states)`. It takes the list of state names a stream produced, such as
`["submitted", "working", "input-required"]`, and returns `"finished"` when the last state is terminal,
`"waiting for the caller"` when it is `input-required` or `auth-required`, and `"still running"` otherwise. Test it on
the two lists in the cell.

In [15]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [16]:
TERMINAL = {"completed", "failed", "rejected", "canceled"}


def final_state(states: list[str]) -> str:
    last = states[-1]
    if last in TERMINAL:
        return "finished"
    if last in ("input-required", "auth-required"):
        return "waiting for the caller"
    return "still running"


print(final_state(["submitted", "working", "input-required"]))
print(final_state(["working", "completed"]))

waiting for the caller
finished


## Stop the agent
The agent runs in its own process. Stop it, close the client and check the port is free again.

In [17]:
await client.close()
server.terminate()
server.wait(timeout=5)
print("port 9270 free again:", not port_open(9270))

port 9270 free again: True


## Recap
- A task has an id, a context id and one state at a time. Four states are final.
- Streaming sends each state change, step message and artifact as it happens.
- Pausing for input keeps the same task: answer with the same task id and context id.
- A failure is a terminal state with a reason; a policy refusal is `rejected`; a retry is a new task.
- Every state needs one caller action. A state without one is a loop that never ends.

**Interview:** "A refund has been `working` for three days. How do you find out what the remote agent is doing, and
what if it died midway?" Strong answer: "I read the task by id: its last state and messages show where it stopped. The
agent stores tasks in a database, so a restart keeps them. For a retry I send the same request with an idempotency key,
so the refund is not paid twice."